# Every match, predicted — then marked

The Dixon-Coles goals model (`ratings.py`) turns 150 years of international results into a
probability for every scoreline of every match. This notebook holds it to account: it
**freezes the model the day before kickoff**, predicts all 72 group fixtures, and then
grades those frozen predictions against the actual results as they land.

Why freezing matters: if the ratings absorbed in-tournament results, the "forecast" would
be cheating. Fitting strictly on matches before `ASOF` makes every call genuinely
out-of-sample. This is the sibling of the Model-vs-Market piece — but the yardstick here is
**reality**, not the bookmaker's closing line.

In [1]:
import pandas as pd

from mlfootball import forecast, ratings, data

pd.set_option("display.width", 110)
print("model frozen at:", forecast.ASOF.date())

model frozen at: 2026-06-10


## 1. Fit the frozen model and predict a match

`outcome_probs` gives 1X2; `score_matrix` gives the full scoreline grid (the most likely
scores come straight off it). Host nations get a venue edge only when playing at home.

In [2]:
s = ratings.fit(data.load(), asof=forecast.ASOF)
fx = forecast.fixtures()
demo = next(f for f in fx if f["home"] == "Brazil")
pred = forecast.predict(s, demo)
print(f"{pred['home']} vs {pred['away']} @ {pred['venue']}")
print(f"  1X2: {pred['p_home']:.0%} / {pred['p_draw']:.0%} / {pred['p_away']:.0%}"
      f"  | xG {pred['exp_home']}–{pred['exp_away']}")
print("  most likely:", [f"{t['h']}-{t['a']} ({t['p']:.0%})" for t in pred["top_scores"]])

Brazil vs Morocco @ MetLife Stadium
  1X2: 40% / 33% / 26%  | xG 1.07–0.81
  most likely: ['0-0 (16%)', '1-0 (16%)', '1-1 (14%)', '0-1 (12%)']


## 2. Pull the actual results

openfootball's public-domain feed marks played matches with a full-time score. We key by
the unordered team pair, so home/away orientation in the feed never matters.

In [3]:
results = forecast.fetch_results()
print(f"{len(results)} matches played so far")

14 matches played so far


## 3. Mark the homework

For every played match: did the model call the right result, how many exact scorelines did
it nail, and does it beat a 33/33/33 coin-flip on Brier and log-loss? Early on this is
noise — an opening round of upsets will sink any model — so the metrics travel with their N.

In [4]:
preds = [forecast.predict(s, f) for f in fx]
sc = forecast.score(preds, results)
print(f"results called right : {sc['n_correct']}/{sc['n_played']} ({sc['pct_correct']:.0%})")
print(f"exact scorelines     : {sc['n_exact']}")
print(f"Brier   model {sc['brier_model']}  vs coin-flip {sc['brier_uniform']}")
print(f"log-loss model {sc['logloss_model']} vs coin-flip {sc['logloss_uniform']}")

results called right : 5/14 (36%)
exact scorelines     : 0
Brier   model 0.7123  vs coin-flip 0.6667
log-loss model 1.1327 vs coin-flip 1.0986


In [5]:
pd.DataFrame(sc["log"])[["date", "home", "away", "pred", "actual", "correct"]].head(14)

,date,home,away,pred,actual,correct
0,2026-06-11,Mexico,South Africa,home,"{'h': 2, 'a': 0}",True
1,2026-06-11,South Korea,Czech Republic,home,"{'h': 2, 'a': 1}",True
2,2026-06-12,Canada,Bosnia and Herzegovina,home,"{'h': 1, 'a': 1}",False
3,2026-06-12,United States,Paraguay,away,"{'h': 4, 'a': 1}",False
4,2026-06-13,Qatar,Switzerland,away,"{'h': 1, 'a': 1}",False
5,2026-06-13,Brazil,Morocco,home,"{'h': 1, 'a': 1}",False
6,2026-06-13,Haiti,Scotland,away,"{'h': 0, 'a': 1}",True
7,2026-06-14,Australia,Turkey,away,"{'h': 2, 'a': 0}",False
8,2026-06-14,Germany,Curaçao,home,"{'h': 7, 'a': 1}",True
9,2026-06-14,Ivory Coast,Ecuador,away,"{'h': 1, 'a': 0}",False


## 4. Emit the bundle

One static `site/data/forecast.json`: per-fixture predictions (1X2, xG, top scores, a 0–5
score grid), the running scorecard, the reliability bins and the prediction log. The page
re-runs this through the tournament; no compute in the browser.

In [6]:
out = forecast.export()
m, scd = out["meta"], out["scorecard"]
print(f"written site/data/forecast.json: {m['n_fixtures']} fixtures (frozen {m['asof']}), "
      f"{scd['n_played']} graded")

written site/data/forecast.json: 72 fixtures (frozen 2026-06-10), 14 graded
